In [ ]:
import os
import sys
import yaml
import tensorflow as tf
import keras
from keras import layers


# import utils functions auto-reload
%load_ext autoreload
%autoreload 2

sys.path.append(os.path.abspath('../src'))

from utils import *
from GPUGuard import GPUGuard

from tensorflow.keras.applications import EfficientNetB5

# Load config (relative to notebooks/)
with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)

SEED = config['seed']
set_seeds(SEED)

In [ ]:
IMG_SIZE = (512, 512)
BATCH_SIZE = 4
NUM_CLASSES = config['num_classes']

# All paths relative to notebooks/
train_dir = config['paths']['train_dir']
val_dir   = config['paths']['val_dir']
test_dir  = config['paths']['test_dir']

train_ds, val_ds, test_ds, class_names = load_datasets(
    train_dir, val_dir, test_dir,
    img_size=IMG_SIZE, batch_size=BATCH_SIZE,
)

# compute class weights from training dataset
class_weights = class_weights(train_ds)

# callbacks
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_f1_macro",
        patience=7,
        restore_best_weights=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_f1_macro",
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1,
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=config['models']['efficientnetb5']['checkpoint'],
        monitor="val_f1_macro",
        save_best_only=True,
        verbose=1,
    ),
    GPUGuard(max_usage_ratio=0.95)
]

In [ ]:
# --- Data Augmentation ---
data_augmentation = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05, fill_mode="nearest"),
        layers.RandomZoom(0.1),
        layers.RandomBrightness(0.1),
        layers.RandomContrast(0.1),
    ], name='data_augmentation')

# --- Build Model ---
base_model = EfficientNetB5(weights="imagenet", include_top=False, input_shape=(*IMG_SIZE, 3))
base_model.trainable = False  # freeze backbone for phase 1

inputs = keras.Input(shape=(*IMG_SIZE, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)     # training=False keeps BN layers in inference mode while frozen
x = layers.GlobalAveragePooling2D()(x)
x = layers.LayerNormalization()(x) 
x = layers.Dense(512, activation="relu", kernel_regularizer=keras.regularizers.l2(0.001))(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = keras.Model(inputs, outputs, name="efficientnetb5_transfer")
model.summary()

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=1e-4, 
        weight_decay=1e-4, 
        gradient_accumulation_steps=32
    ),
    loss="categorical_crossentropy", 
    metrics=["accuracy", keras.metrics.F1Score(average="macro", name="f1_macro")],
)

In [ ]:
history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks,
    class_weight=class_weights
)

In [ ]:
# --- Phase 2: Fine-tuning ---

# Unfreeze the last ~50 layers of EfficientNetB5.
# EfficientNet blocks are deeper than ResNet, so more layers must be unfrozen
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

# Very low LR to avoid disrupting pretrained weights
model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=1e-5,
        weight_decay=1e-5,
        gradient_accumulation_steps=32
        ),
    loss="categorical_crossentropy",
    metrics=["accuracy", keras.metrics.F1Score(average="macro", name="f1_macro")],
)

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks,
    class_weight=class_weights,
)

In [ ]:
test_loss, test_accuracy, test_f1 = predict_tta(model, test_ds)
print(f"TTA Results - Loss: {test_loss:.4f} | Accuracy: {test_accuracy:.4f} | F1: {test_f1:.4f}")


In [ ]:
plot_learning_curves([history_phase1, history_phase2], title="EfficientNetB5 Transfer Learning")

In [ ]:
metrics_efficientnet = evaluate_model(model, test_ds, class_names, "EfficientNetB5 Transfer Learning")

In [ ]:
save_history([history_phase1, history_phase2], config['models']['efficientnetb5']['history'])